# 01data-process 노트북 목표
1. 엑셀 파일 불러오기
2. 데이터 전처리
3. csv 형태로 저장


## 1. 데이터 전처리 사전 준비

- reviews 폴더에 크롤링된 엑셀 데이터를 저장한다.
- 각 데이터의 주요 컬럼은 다음과 같다.
  - URL: 리뷰가 등록된 가게 URL
  - 가게명: 가게 이름
  - RATING: 별점
  - CONTENT: 리뷰 본문
  - MENU: 주문 메뉴
  - IMAGE_URLS: 리뷰 이미지 URL

## 2. 라이브러리 로드

- 원본 엑셀 파일을 불러오고, 리뷰 텍스트를 정제하며, 분석용 메타데이터를 만들기 위한 패키지를 임포트한다.
- 필요한 패키지를 한 번에 확인해 실행 환경 차이로 인한 오류를 줄이기 위해서다.
- 각 패키지의 역할은 다음과 같다.
    1. `Path`: `reviews/` 폴더 안의 엑셀 파일을 탐색한다.
    2. `re`: 리뷰 이벤트 키워드 탐지와 텍스트 정제를 위한 정규표현식 처리에 사용한다.
    3. `os`: 전처리 결과 CSV가 실제로 저장되었는지 확인한다.
    4. `pandas`: 엑셀/CSV 파일을 읽고 데이터프레임 형태로 처리한다.
    5. `emoji`: 리뷰 텍스트에서 이모지를 제거하고, 이모지 개수를 계산한다.
    6. `repeat_normalize`: `ㅋㅋㅋㅋ`, `ㅠㅠㅠㅠ`처럼 반복되는 문자를 정규화한다.

In [1]:
from pathlib import Path
import re
import os
import pandas as pd

import emoji
from soynlp.normalizer import repeat_normalize

## 3. 엑셀 파일 로드

- `reviews` 폴더 내 모든 `.xlsx` 및 `.xls` 파일을 로드하여 하나의 데이터프레임으로 병합한다.
- 일부 해시스크래퍼 엑셀 파일은 첫 행에 `무료 체험 기간 중 최대 100개의 결과만 다운로드 가능합니다.` 같은 안내 문구가 들어 있으므로, 해당 경우에 첫 행을 제거한다.
- 첫 행부터 실제 리뷰가 들어있는 파일은 첫 행을 유지한다. 
- 각 행이 어떤 엑셀 파일에서 왔는지 추적할 수 있도록 `source_file` 컬럼을 추가한다.


In [2]:
def is_hashscraper_notice_row(row):
    notice_keywords = [
        "free trial",
        "max 100",
        "download",
        "무료 체험 기간",
        "최대 100개의 결과",
        "다운로드 가능합니다",
    ]
    row_text = " ".join(row.fillna("").astype(str).tolist())
    return any(keyword in row_text for keyword in notice_keywords)


def load_excel_files(folder_path):
    folder_path = Path(folder_path)
    excel_files = sorted(list(folder_path.rglob("*.xlsx")) + list(folder_path.rglob("*.xls")))

    if len(excel_files) == 0:
        raise FileNotFoundError(f"엑셀 파일을 찾을 수 없습니다: {folder_path}")

    df_list = []
    skipped_notice_rows = []

    for file_path in excel_files:
        temp_df = pd.read_excel(file_path)

        if len(temp_df) > 0 and is_hashscraper_notice_row(temp_df.iloc[0]):
            temp_df = temp_df.iloc[1:].reset_index(drop=True)
            skipped_notice_rows.append(file_path.name)

        temp_df["source_file"] = file_path.name
        df_list.append(temp_df)

    return pd.concat(df_list, ignore_index=True)


df = load_excel_files("reviews")

# 로드한 엑셀 파일의 shape
print("로드 완료:", df.shape)

# 최초 데이터 형태 저장
original_shape_of_data= df.shape

# 첫 3행 출력
df.head(3)

로드 완료: (11310, 11)


,URL,NAME,RATING,CONTENT,MENU,DATE,IMAGE_URLS,OWNER_REPLY,created_at,source_file,가게명
0,https://s.baemin.com/pG000flG524ai,송청원,1.0,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"이번 주, 알뜰배달","[""https://bmreview.cdn.baemin.com/bmreview-qh2...","송청원님, 저희 기장국수를 찾아와 주셔서 감사합니다 하지만 이번 주문시 늦게 배달되...",2026-05-02 20:28:40 +0900,배달의민족_리뷰_수집_부산.xlsx,NaN
1,https://s.baemin.com/pG000flG524ai,철수,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"1개월 전, 가게배달",NaN,"철수님, 저희 기장국수를 찾아와 주셔서 감사합니다 하지만 이번 주문시 늦게 배달되...",2026-05-02 20:28:40 +0900,배달의민족_리뷰_수집_부산.xlsx,NaN
2,https://s.baemin.com/pG000flG524ai,쓰윽쓱,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","2개월 전, 한집배달","[""https://bmreview.cdn.baemin.com/bmreview-qh2...","쓰윽쓱님, 저희 기장국수를 찾아와 주셔서 감사합니다 하지만 이번 주문시 국물이 새어...",2026-05-02 20:28:41 +0900,배달의민족_리뷰_수집_부산.xlsx,NaN


## 4. 컬럼 정리

- 한글 컬럼 -> 영어로 변환
- 안쓰는 컬럼 삭제
- 원본 컬럼명이 파일마다 다를 수 있어, 이후 단계에서 같은 이름으로 접근 가능


In [3]:
# 일부 컬럼을 변경
df = df.rename(columns={
    "URL": "store_url",
    "가게명": "store_name",
    "RATING": "rating",
    "CONTENT": "review_text",
    "MENU": "menu_name",
    "IMAGE_URLS": "image_urls",
})

# 프로젝트에서 사용하는 컬럼만 유지
use_cols = [
    "store_url",
    "store_name",
    "rating",
    "review_text",
    "menu_name",
    "image_urls",
    "source_file"
]

df = df[use_cols]

# 출력
print(df.shape)
df.head(3)

(11310, 7)


,store_url,store_name,rating,review_text,menu_name,image_urls,source_file
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx


## 5. 기본 전처리

1. `review_text_raw`: 원문 리뷰 텍스트 보존
2. `menu_name_raw`: 원문 메뉴명 보존
3. 리뷰 텍스트가 비어 있는 행 제거
4. 별점(`rating`)을 숫자형 데이터로 변환

In [4]:
df["review_text_raw"] = df["review_text"].fillna("").astype(str)
df["menu_name_raw"] = df["menu_name"].fillna("").astype(str)

df = df[df["review_text_raw"].str.strip() != ""].reset_index(drop=True)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

print(df.shape)
df.head(3)

(9128, 9)


,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥"


## 6. 중복제거
- 크롤링되면서 같은 데이터인 경우를 삭제


In [5]:
df = df.drop_duplicates(
    subset=["store_url", "store_name", "review_text_raw", "menu_name_raw"]
).reset_index(drop=True)

print(df.shape)
df.head(3)

(8841, 9)


,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥"


## 7. 리뷰 이벤트 여부 판단 후 레이블화

- `menu_name_raw`에 리뷰 이벤트 전용 메뉴 또는 리뷰 약속 문구가 포함된 경우 `label=1`로 설정한다.
- 그 외의 경우는 `label=0`으로 설정한다.
- re.IGNORECASE를 통해 대소문자 구분하지 않는다.


In [6]:
event_menu_pattern = re.compile(
    r"(리뷰\s*이벤트|리뷰이벤트|리뷰\s*참여|리뷰참여|리뷰\s*약속|리뷰약속|review\s*event|ri\s*뷰|니\s*뷰|행복\s*음료|500원의\s*행복|아메리카노/매실/아이스티|rㅣ뷰\s*이벤트|rㅣ뷰\s*eㅣ벤트|포토\s*리뷰|리뷰\s*서비스|리뷰\s*할인|\(리뷰\)|리뷰두\s*찜두|100원의\s*행복|백원의\s*행복)",
    flags=re.IGNORECASE
)

def label_review_event_from_menu(menu):
    menu = str(menu)
    return 1 if event_menu_pattern.search(menu) else 0

df["label"] = df["menu_name_raw"].apply(label_review_event_from_menu)

print(df.shape)
df.head(3)

(8841, 10)


,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw,label
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥",0


## 8. KcBERT 입력을 위한 텍스트 정제 및 메타데이터 생성
- https://github.com/Beomi/KcBERT
- 깃허브 링크 내부의 이모지 관련 체크
- 메타데이터는 다음과 같다.
    1. 리뷰 텍스트 길이
    2. 이모지 개수
    3. 사진 개수
- KcBERT에는 정제 텍스트가 필요하고, MLP에는 텍스트 + 메타데이터 학습할 예정이다.
- 별점은 이후에 별점 정제에 사용할 값으로, 메타데이터 후보에서 제거한다.


In [7]:
# KcBERT를 위한 전처리
pattern = re.compile(r'[^ .,?!/@$%~％·∼()\x00-\x7F ㄱ-ㅣ가-힣]+')

url_pattern = re.compile(
    r'https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}'
    r'\.[a-zA-Z0-9()]{1,6}\b'
    r'([-a-zA-Z0-9()@:%_\+.~#?&//=]*)'
)

def clean_review_text(text):
    text = str(text)
    text = url_pattern.sub("", text)
    text = emoji.replace_emoji(text, replace="")
    text = pattern.sub(" ", text)
    text = text.strip()
    text = repeat_normalize(text, num_repeats=2)
    return text

df["cleaned_review_text"] = df["review_text_raw"].apply(clean_review_text)

# 메타데이터 생성
def count_photos(image_urls):
    if pd.isna(image_urls):
        return 0

    image_urls = str(image_urls).strip()

    if image_urls == "" or image_urls in ["[]", "None", "nan", "NaN"]:
        return 0

    return image_urls.count("http")


df["photo_count"] = df["image_urls"].apply(count_photos)

def count_emoji(text):
    return len([char for char in str(text) if char in emoji.EMOJI_DATA])

df["text_length"] = df["review_text_raw"].apply(len)
df["emoji_count"] = df["review_text_raw"].apply(count_emoji)

print(df.shape)
df.head(3)

(8841, 14)


,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw,label,cleaned_review_text,photo_count,text_length,emoji_count
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0,좋아하는집인데 배달은 안하시는게... 떡국이아니라 떡죽이네요...,1,36,0
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,0,16,0
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥",0,이걸 수습하기가 참.. 귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러서...,2,90,0


## 9. 데이터 저장
- 저장, 작업 폴더 안에 생성됨
- 이후 KcBERT에 입력으로 사용가능
- index=False로 행번호는 저장하지 않는다.
- encoding="utf-8-sig로 한글 깨짐을 줄인다.
- 이후 저장 여부 파악 후, Ture, False 출력


In [8]:
df.to_csv("csv/preprocessed_reviews.csv", index=False, encoding="utf-8-sig")
print(os.path.exists("csv/preprocessed_reviews.csv"))

True


## 10. 저장된 CSV 테스트

In [9]:
df_saved = pd.read_csv("csv/preprocessed_reviews.csv", encoding="utf-8-sig")

print("리뷰 개수:", len(df_saved))
print("컬럼 개수:", len(df_saved.columns))
print("데이터 shape:", df_saved.shape)
print("리뷰 이벤트 개수:", (df_saved["label"] == 1).sum())
print("일반 리뷰 개수:", (df_saved["label"] == 0).sum())

df_saved.head(3)

리뷰 개수: 8841
컬럼 개수: 14
데이터 shape: (8841, 14)
리뷰 이벤트 개수: 3150
일반 리뷰 개수: 5691


,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw,label,cleaned_review_text,photo_count,text_length,emoji_count
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,좋아하는집인데 배달은 안하시는게...\n떡국이아니라 떡죽이네요...,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0,좋아하는집인데 배달은 안하시는게... 떡국이아니라 떡죽이네요...,1,36,0
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,0,16,0
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥",0,이걸 수습하기가 참.. 귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러서...,2,90,0


## 11. 분석
- 표 형태로 분석


In [10]:
# 1. 일반 리뷰와 이벤트 리뷰 분리 (label: 0=일반, 1=이벤트)
normal_df = df_saved[df_saved["label"] == 0]
event_df = df_saved[df_saved["label"] == 1]

# 2. 데이터 수 및 전체 데이터 수 계산
normal_count = len(normal_df)
event_count = len(event_df)
total_count = len(df_saved)

# 3. 데이터프레임으로 결과 표 구성
result_data = {
    "항목": [
        "데이터 수",
        "비율",
        "평균 리뷰 길이",
        "평균 이모지 수",
        "평균 사진 수"
    ],
    "일반 리뷰": [
        f"{normal_count:,}",
        f"{(normal_count / total_count * 100):.1f}%",
        f"{normal_df['text_length'].mean():.1f}",
        f"{normal_df['emoji_count'].mean():.1f}",
        f"{normal_df['photo_count'].mean():.1f}"
    ],
    "이벤트 리뷰": [
        f"{event_count:,}",
        f"{(event_count / total_count * 100):.1f}%",
        f"{event_df['text_length'].mean():.1f}",
        f"{event_df['emoji_count'].mean():.1f}",
        f"{event_df['photo_count'].mean():.1f}"
    ]
}

result_table = pd.DataFrame(result_data)

# 결과 출력
print("=== 리뷰 분석 결과 표 ===")
print(result_table.to_string(index=False))

original_row_count = original_shape_of_data[0]
final_row_count = len(df_saved)
removed_count = original_row_count - final_row_count
removed_rate = removed_count / original_row_count * 100

print("\n");

print("=== 데이터 변화 ===")
print(f"전체 리뷰 수: {original_row_count:,} -> {final_row_count:,}")
print(f"제거된 리뷰 수: {removed_count:,}개 ({removed_rate:.1f}%)")


=== 리뷰 분석 결과 표 ===
      항목 일반 리뷰 이벤트 리뷰
   데이터 수 5,691  3,150
      비율 64.4%  35.6%
평균 리뷰 길이  57.7   54.7
평균 이모지 수   0.7    0.6
 평균 사진 수   0.9    1.0


=== 데이터 변화 ===
전체 리뷰 수: 11,310 -> 8,841
제거된 리뷰 수: 2,469개 (21.8%)
